# LAB3 — Gom cụm Mall Customers (trang 18–19)

**Sinh viên:** Trần Chí Nguyên — **MSSV:** 6451071052.

> Mỗi câu lập trình in họ tên và MSSV theo template. Chạy lần lượt từ trên xuống.

In [1]:
%matplotlib inline
HO_TEN = 'Trần Chí Nguyên'
MSSV = '6451071052'
def in_danh_tinh(): print(f'{HO_TEN} - {MSSV}')


## Câu 1. Nhập dữ liệu, thống kê và xử lý thiếu

Tải Mall Customers CSV từ Google Drive, thống kê và điền giá trị thiếu (median cho số, mode cho phân loại).

In [2]:
in_danh_tinh()
import io, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
DATA_URL='https://drive.google.com/uc?export=download&id=1bsPcz8nZOMPcF3eVVs-EfmDRsqfjjUcC'
with urllib.request.urlopen(DATA_URL,timeout=60) as response:
    df=pd.read_csv(io.BytesIO(response.read()))
print('Shape:',df.shape); display(df.head()); display(df.info())
print('Thiếu trước xử lý:\n',df.isna().sum())
for col in df.select_dtypes(include=np.number): df[col]=df[col].fillna(df[col].median())
for col in df.select_dtypes(exclude=np.number):
    if df[col].isna().any(): df[col]=df[col].fillna(df[col].mode().iloc[0])
print('Thiếu sau xử lý:\n',df.isna().sum()); display(df.describe(include='all').T)

Trần Chí Nguyên - 6451071052


Shape: (200, 5)


,CustomerID,Gender,Age,Annual Income (k$),Spending Score (1-100)
0,1,Male,19,15,39
1,2,Male,21,15,81
2,3,Female,20,16,6
3,4,Female,23,16,77
4,5,Female,31,17,40


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CustomerID              200 non-null    int64 
 1   Gender                  200 non-null    object
 2   Age                     200 non-null    int64 
 3   Annual Income (k$)      200 non-null    int64 
 4   Spending Score (1-100)  200 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 7.9+ KB


None

Thiếu trước xử lý:
 CustomerID                0
Gender                    0
Age                       0
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64
Thiếu sau xử lý:
 CustomerID                0
Gender                    0
Age                       0
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
CustomerID,200.0,NaN,NaN,NaN,100.5,57.879185,1.0,50.75,100.5,150.25,200.0
Gender,200,2,Female,112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,200.0,NaN,NaN,NaN,38.85,13.969007,18.0,28.75,36.0,49.0,70.0
Annual Income (k$),200.0,NaN,NaN,NaN,60.56,26.264721,15.0,41.5,61.5,78.0,137.0
Spending Score (1-100),200.0,NaN,NaN,NaN,50.2,25.823522,1.0,34.75,50.0,73.0,99.0


## Câu 2–3. Histogram và số lượng theo giới tính

Vẽ phân phối Age, Annual Income, Spending Score; thống kê số dòng theo Gender.

In [3]:
in_danh_tinh()
cols=['Age','Annual Income (k$)','Spending Score (1-100)']
fig,axes=plt.subplots(1,3,figsize=(14,4))
for c,ax in zip(cols,axes): sns.histplot(df[c],kde=True,ax=ax); ax.set_title(c)
plt.tight_layout(); plt.show()
print(df['Gender'].value_counts(dropna=False))
sns.countplot(data=df,x='Gender'); plt.title('Số khách theo giới tính'); plt.show()

Trần Chí Nguyên - 6451071052


Gender
Female    112
Male       88
Name: count, dtype: int64


  plt.tight_layout(); plt.show()
  sns.countplot(data=df,x='Gender'); plt.title('Số khách theo giới tính'); plt.show()


## Câu 4–5. Scatter plot theo giới tính

Khảo sát Age–Annual Income và Annual Income–Spending Score, tô màu theo Gender.

In [4]:
in_danh_tinh()
fig,axes=plt.subplots(1,2,figsize=(12,5))
sns.scatterplot(data=df,x='Age',y='Annual Income (k$)',hue='Gender',ax=axes[0])
sns.scatterplot(data=df,x='Annual Income (k$)',y='Spending Score (1-100)',hue='Gender',ax=axes[1])
plt.tight_layout(); plt.show()

Trần Chí Nguyên - 6451071052


  plt.tight_layout(); plt.show()


## Câu 6–8. Elbow, k-Means và Silhouette cho Age và Spending Score

Tính inertia với k=1..10, chọn k ở điểm gối bằng quan sát đồ thị; bộ Mall thường cho điểm gối gần 4. Chuẩn hóa trước khi gom cụm.

In [5]:
in_danh_tinh()
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from IPython.display import display
from sklearn.metrics import pairwise_distances

def cluster_and_plot(features,k,title):
    scaler=StandardScaler(); Z=scaler.fit_transform(df[features])
    inertias=[]
    for n in range(1,11): inertias.append(KMeans(n_clusters=n,random_state=42,n_init=10).fit(Z).inertia_)
    plt.plot(range(1,11),inertias,marker='o'); plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow — '+title); plt.grid(); plt.show()
    model=KMeans(n_clusters=k,random_state=42,n_init=10); labels=model.fit_predict(Z)
    print(f'{title}: k={k}, silhouette={silhouette_score(Z,labels):.4f}')
    return Z,labels,model,scaler

features_age=['Age','Spending Score (1-100)']
K_AGE_SCORE=4
Z_age,labels_age,km_age,scale_age=cluster_and_plot(features_age,K_AGE_SCORE,'Age & Spending Score')
df['Cluster_Age_Spending']=labels_age
sns.scatterplot(x=df[features_age[0]],y=df[features_age[1]],hue=labels_age,palette='tab10'); plt.title('k-Means: Age & Spending Score'); plt.show()

Trần Chí Nguyên - 6451071052


Age & Spending Score: k=4, silhouette=0.4384


  plt.plot(range(1,11),inertias,marker='o'); plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow — '+title); plt.grid(); plt.show()
  sns.scatterplot(x=df[features_age[0]],y=df[features_age[1]],hue=labels_age,palette='tab10'); plt.title('k-Means: Age & Spending Score'); plt.show()


## Câu 9. Lặp lại cho Income–Spending và ba thuộc tính

Vẽ elbow, phân cụm, silhouette cho hai cột rồi ba cột. Biểu diễn ba thuộc tính bằng scatter 3D. Chọn k theo elbow của biểu đồ tương ứng.

In [6]:
in_danh_tinh()
from mpl_toolkits.mplot3d import Axes3D
features_income=['Annual Income (k$)','Spending Score (1-100)']
K_INCOME_SCORE=5
Z_inc,labels_inc,km_inc,scale_inc=cluster_and_plot(features_income,K_INCOME_SCORE,'Income & Spending')
df['Cluster_Income_Spending']=labels_inc
sns.scatterplot(x=df[features_income[0]],y=df[features_income[1]],hue=labels_inc,palette='tab10'); plt.title('k-Means: Income & Spending'); plt.show()
features_3=['Age','Annual Income (k$)','Spending Score (1-100)']
K_THREE_FEATURES=6
Z3,labels3,km3,scale3=cluster_and_plot(features_3,K_THREE_FEATURES,'Age, Income & Spending')
df['Cluster_3D']=labels3
fig=plt.figure(figsize=(9,7)); ax=fig.add_subplot(111,projection='3d')
sc=ax.scatter(df[features_3[0]],df[features_3[1]],df[features_3[2]],c=labels3,cmap='tab10')
ax.set_xlabel(features_3[0]); ax.set_ylabel(features_3[1]); ax.set_zlabel(features_3[2]); ax.set_title('k-Means 3D'); plt.colorbar(sc); plt.show()
print('Silhouette income/spending:',silhouette_score(Z_inc,labels_inc))
print('Silhouette 3D:',silhouette_score(Z3,labels3))

Trần Chí Nguyên - 6451071052


  plt.plot(range(1,11),inertias,marker='o'); plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow — '+title); plt.grid(); plt.show()
  sns.scatterplot(x=df[features_income[0]],y=df[features_income[1]],hue=labels_inc,palette='tab10'); plt.title('k-Means: Income & Spending'); plt.show()


Income & Spending: k=5, silhouette=0.5547


Age, Income & Spending: k=6, silhouette=0.4284
Silhouette income/spending: 0.5546571631111091
Silhouette 3D: 0.4284167762892593


  plt.plot(range(1,11),inertias,marker='o'); plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow — '+title); plt.grid(); plt.show()
  ax.set_xlabel(features_3[0]); ax.set_ylabel(features_3[1]); ax.set_zlabel(features_3[2]); ax.set_title('k-Means 3D'); plt.colorbar(sc); plt.show()
